# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5 (2,3) / Выводится матрица 2 строки 3 столбца. a[1,2] это 5 , а.shape это размерность матрицы нашей.  
2. 1 8 / a.argmax это индекс максимального значения, берет первое встретившиеся максимальное, a.max это максимальное значение
3. 99 / срез b это не отдельная копия, меняем b меняется a
4. 2 0.5 / .sum() два значения подходят под фильтр a>3 , они становятся True это единица, 1+1=2. .mean() среднее значение(доля) 2/4=0.5 
5. (2,3) 0.25 / Размерность матрицы , создается одномерный массив данных от 0 до 1, разбитый на 5 равных частей, [1] индекс 0.25

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
# База

In [4]:
top_idx = amount.argmax()
print(top_idx, amount[top_idx], cust[top_idx], city[top_idx])

1001 9999990.0 TEST Алматы


Самый дорогой заказ под индексом 1001 на сумму 9 999 990 ₸ сделал клиент TEST из Алматы

In [9]:
print(amount[paid].sum(), amount[~paid].sum())

101909360.0 7974000.0


101909360.0 + выручка оплаченных заказов

7974000.0 + сумма неоплаченных заказов

In [14]:
shim = city == "Шымкент"
print(shim.sum(), shim.mean())

804 0.1471718835804503


804 + оплаченных заказов в Шымкенте

14.7% доля заказов оформленных в Шымкенте

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [6]:
discount_all = amount[(paid) & (amount > 50000)].sum() * 0.15
print(discount_all)

5702422.5


In [7]:
without_test = amount[(paid) & (amount > 50000) & (cust != 'TEST')].sum() * 0.15
print(without_test)

1202427.0


Ответы различаются почти в 4.7 раза, потому что у клиента TEST было 3 аномально крупных тестовых заказа по 9.99 млн ₸ каждый. При расчете скидки 15% только эти 3 заказа дают 4.5 млн ₸ недополученной выручки, сильнейше искажая общую картину.

In [13]:
cities = np.unique(city)
city_shares = {}
for c in cities:
    mask = (city == c) & (~np.isnan(days)) #создал маску для города с отсечением пропусков
    share = (days[mask] <= 2).mean() #подсчет доли быстрой доставки по этой маске
    city_shares[c] = share
for c, share in city_shares.items():
    print(f"{c}: {round(share * 100, 1)}%")

Актобе: 62.8%
Алматы: 66.5%
Астана: 65.4%
Атырау: 66.7%
Караганда: 71.0%
Шымкент: 30.7%


In [10]:
avg = amount.mean()
print("Средний чек:", round(avg))

Средний чек: 20114


In [11]:
mask = (paid) & (cust != 'TEST')
correct_avg = amount[mask].mean()
print("Настоящий средний чек:", round(correct_avg))

Настоящий средний чек: 14672


Ошибка 1: средний чек это только оплаченные заказы, а не все подряд(возврат, оплаченные, отмененные). 
Ошибка 2: ИИ не убрал 3 тестовых заказа, где каждый заказ по 10 млн тг, это сильно искажает данные.    

In [ ]:
# Стандарт


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж